# Crater annotation build audit

## tl;dr

The current artifact is `data_orbital/crater_annotations_v2`: 100 images, 400 Q1–Q4 candidates, 264 invariant candidate answers and 136 unknown references. Scoring targets are intentionally empty until explicit second review. This is reference preparation, not model performance.

## Context & Methods

Read-only integrity, count reconciliation, source-instance audit and inference/GT separation checks. No inference, downloading, training or source writes.

### Key Assumptions

CVAT full-image review tags are not final GT acceptance. Source polygon raster equality is alignment evidence, not independent geological validation. Semantic components are never crater instances. The plain Python cells were executed sequentially with captured outputs; a Jupyter kernel was not installed or used. The exact build converter is preserved under `converter_snapshot/`.

In [1]:
from pathlib import Path
import json, sys
from collections import Counter
WORKSPACE=Path('C:\\Users\\Timur\\myprojects\\robo-projects\\article')
BUILD=WORKSPACE/'data_orbital/crater_annotations_v2'
sys.path.insert(0,str(WORKSPACE/'src'))
from planetary_vlm.datasets.crater_annotations import verify_crater_build
from planetary_vlm.io import read_jsonl
verification=verify_crater_build(WORKSPACE,BUILD)
print(json.dumps(verification,indent=2))

{
  "schema": "crater-annotations-v1",
  "integrity_verified": true,
  "bundle_sha256": "27153ee4782afc384d701a1a205a28820218b6fe260998bb9b1c899a0d87bb88",
  "images": 100,
  "candidate_questions": 400,
  "reference_ready_questions": 0,
  "ready_for_scored_inference": false,
  "scientific_result": false
}


## Data

The build snapshots four supplied CVAT ZIPs, selected source annotation provenance, native image/mask hashes, original splits/groups/licenses and converter hashes. The source-only cache contains four pinned lunar COCO files and a pinned Mars metadata ZIP; train CSV members are not read for this build.

In [2]:
metadata=json.loads((BUILD/'metadata.json').read_text(encoding='utf-8'))
samples=read_jsonl(BUILD/'samples.jsonl')
instances=read_jsonl(BUILD/'instances.jsonl')
questions=read_jsonl(BUILD/'questions.jsonl')
catalog=read_jsonl(BUILD/'source_catalog_pending.jsonl')
assert len(samples)==metadata['images']==100
assert len(instances)==2074
assert len(catalog)==1101
assert len([o for o in instances if o['origin']=='publisher_coco'])==1816
assert len([o for o in instances if o['origin']=='human_CVAT_supplement'])==258
print('Samples by planet / publisher split:',dict(Counter((s['planet'],s['source_split']) for s in samples)))
print('Reconstructed source-mask matches:',sum(s['source_annotation_audit']['source_mask_reproduced_exactly'] is True for s in samples))
print('Mars frames with pending source geometry:',sum(not s['source_annotation_audit']['source_instances_complete'] for s in samples))

Samples by planet / publisher split: {('mars', 'test'): 33, ('mars', 'val'): 17, ('moon', 'val'): 33, ('moon', 'test'): 17}
Reconstructed source-mask matches: 29
Mars frames with pending source geometry: 26


## Results

Candidate references and unknowns are reported separately. A pending visibility concern must invalidate that object as an existential certificate, not merely add an unknown extra object.

In [3]:
known=[q for q in questions if q['candidate_answer'] is not None]
unknown=[q for q in questions if q['reference_status']=='UNKNOWN_REFERENCE']
assert len(questions)==400 and len(known)==264 and len(unknown)==136
assert all(q['reference_answer'] is None for q in questions)
assert read_jsonl(BUILD/'targets.jsonl')==[]
assert read_jsonl(BUILD/'requests.jsonl')==[]
assert not verification['ready_for_scored_inference']
for sid in ['mars_seg_033','mars_seg_046']:
    q1=next(q for q in questions if q['sample_id']==sid and q['task_id']=='crater_presence_v1')
    assert q1['candidate_answer'] is None
print('Candidate answers by planet:',dict(Counter(q['planet'] for q in known)))
print('Unknown reasons:',dict(Counter(q['admission_reason'] for q in unknown)))

Candidate answers by planet: {'mars': 88, 'moon': 176}
Unknown reasons: {'source_instances_incomplete': 100, 'source_instances_incomplete;unknown_visual_regions': 4, 'existing_object_visibility_or_eligibility_unresolved;unknown_visual_regions': 11, 'uncertain_class_or_geometry_unbounded': 21}


In [4]:
from planetary_vlm.datasets.manifest import request_from_dict
from planetary_vlm.datasets.crater_reference import prompt_for
index={s['sample_id']:s for s in samples}
by_request={q['request_id']:q for q in questions}
requests=read_jsonl(BUILD/'candidate_requests.jsonl')
assert len(requests)==400
for raw in requests:
    request=request_from_dict(raw)
    question=by_request[request.request_id]
    sample=index[question['sample_id']]
    assert not request.modality_paths and not request.do_sample
    assert request.image_paths==(sample['image'],)
    assert request.prompt==prompt_for(question['task_id'],*sample['native_size'])
    assert request.allowed_answers==tuple(question['allowed_answers'])
print('400 strict RGB-only candidate ModelRequests; no masks, correct answers or outcome telemetry.')

400 strict RGB-only candidate ModelRequests; no masks, correct answers or outcome telemetry.


## Takeaways

The technical build is reproducible and preserves source splits and immutable originals. It is not ready for scored inference: 26 Mars source frames need a verified native-pixel geometry mapping, and per-image hash-bound second review must account for uncertainty, overlapping identities and visibility. Use `review_queue.jsonl` as a template in a separate decisions file, then build a new version with `--reviews`; never auto-accept this queue.

The schema/converter remains version 1; folder revision v2 supersedes the unscored intermediate folder v1. Full instructions: `docs/protocols/crater_annotations_v1.md`.